<a href="https://colab.research.google.com/github/rendzina/BigDataAndVisualisation/blob/main/Colab/Hello_World_CoLab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hello World
## Google Colab

This notebook mounts **Azure Blob** storage, reads **HVAC** sample data into a Spark **DataFrame**, explores it, registers a **temporary view**, and runs **SQL** with `spark.sql`. A short overview below explains how Spark behaves on Colab; then you **inspect your session** before loading the CSV.

## How Spark works on Google Colab (reminder)

Colab gives you a **single Linux VM**. Spark still treats work as **parallel**: with `local[*]` the session can use **all CPU cores** on that machine. The notebook acts as the **driver**; Spark schedules **tasks** over **partitions** of the data (here, rows read from the CSV on your mounted storage).

**Data is partitioned.** Even on one VM, a DataFrame is split into **partitions** so Spark can use multiple cores when it runs actions.

An **RDD** (**resilient distributed dataset**) is simply a dataset **split into partitions** across Spark so work can run **in parallel** (on Colab, mainly across cores on the same VM).

**Lazy evaluation:** calls such as `read.csv` usually record a **plan** first; heavy work runs when you call an **action** (for example `count()`, `show()`, or `spark.sql(...).show()`).

### Mount Azure Blob storage

The sample data lives in Azure Blob storage. We install the mount package and then mount the storage so the notebook can read the HVAC CSV file.

**Install the mount-azure-blob package** (run once per session).

**Mount the storage** using the connection details given in class. You will be prompted for account name and key.

In [ ]:
%%capture
%pip install mount-azure-blob==0.0.3

In [ ]:
# Connection details are given in class
from mount_azure_blob import mount_storage
mount_storage(mount_path="bdv-2024-05-09t15-59-02-855z", config_file=None)

### Initialise Spark

Google Colab already includes PySpark. Create a `SparkSession` with `local[*]` so all CPU cores on the VM can be used. This is the entry point for DataFrames and SQL.

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("BDV Spark Example")
    .master("local[*]")
    .getOrCreate()
)
spark


## Inspect your Spark session

On **Google Colab**, **master** is usually `local[*]` because Spark runs on the Colab VM using all CPU cores. **App name** identifies this session in the Spark UI if you open it.

In [ ]:
print("Spark version:", spark.version)
sc = spark.sparkContext
print("Master:", sc.master)
print("App name:", sc.appName)


**Read the HVAC CSV** from the mounted blob path into a Spark DataFrame. We use the first row as the header and let Spark infer column types.

In [ ]:
# Create a Spark dataframe and table from sample data (note this is NOT a Pandas dataframe)
csvFile = spark.read.csv('bdv-2024-05-09t15-59-02-855z/HdiSamples/HdiSamples/SensorSampleData/hvac/HVAC.csv', header=True, inferSchema=True)

**Row count:** check how many records are in the DataFrame.

In [ ]:
# How many rows in dataframe?
csvFile.count()

### HVAC dataset description

The **HVAC** (Heating, Ventilation, and Air Conditioning) sample data contains sensor readings from buildings: date and time, target and actual temperatures, system identifier, system age, and building ID. We use it to explore DataFrames and later to run SQL queries.

In [ ]:
# Show the schema: column names and types inferred by Spark
csvFile.printSchema()

In [ ]:
# Preview the first 10 rows
csvFile.show(10)

In [ ]:
# Summary statistics for numeric columns
csvFile.describe().show()

### Register the DataFrame as a SQL temp view

We register `csvFile` as a temporary view so we can query it with **`spark.sql("...").show()`** in the following cells. That uses the same SQL engine as the DataFrame API.

In [ ]:
csvFile.createOrReplaceTempView("hvac_ss01shh")

**Verify row count** using SQL (should match the DataFrame count above).

In [ ]:
spark.sql("SELECT count(*) from hvac_ss01shh").show()

**Describe the table** to list columns and their types in SQL catalog form.

In [ ]:
spark.sql("DESCRIBE TABLE hvac_ss01shh").show()


**Example query:** temperature difference (target minus actual) for a given date. Positive values mean the system was heating; negative means cooling.


In [ ]:
query = """
SELECT `buildingID`, (targettemp - actualtemp) AS temp_diff, date
FROM hvac_ss01shh
WHERE date = '6/1/13'
"""
spark.sql(query).show()
